In [1]:
!pip install gymnasium stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 5.0 MB/s eta 0:00:00


In [2]:
import gymnasium as gym
from gymnasium import spaces

import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("processed_data.csv")

df.head()
df.info()
print(df.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3323 entries, 0 to 3322
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Crop             3323 non-null   object 
 1   Crop_Year        3323 non-null   int64  
 2   Season           3323 non-null   object 
 3   State            3323 non-null   object 
 4   Area             3323 non-null   float64
 5   Production       3323 non-null   int64  
 6   Annual_Rainfall  3323 non-null   float64
 7   Fertilizer       3323 non-null   float64
 8   Pesticide        3323 non-null   float64
 9   Yield            3323 non-null   float64
 10  Commodity        3323 non-null   object 
 11  Modal_Price      3323 non-null   float64
 12  Revenue          3323 non-null   float64
 13  Cost             3323 non-null   float64
 14  Profit           3323 non-null   float64
dtypes: float64(9), int64(2), object(4)
memory usage: 389.5+ KB
Index(['Crop', 'Crop_Year', 'Season', 'State', 

In [7]:
# Number of unique crops
print("Number of Crops:", df["Crop"].nunique())

# List all crops
print(df["Crop"].unique())

Number of Crops: 6
['Cotton' 'Maize' 'Onion' 'Potato' 'Wheat' 'Banana']


In [14]:
print(df.columns.tolist())

['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield', 'Commodity', 'Modal_Price', 'Revenue', 'Cost', 'Profit']


In [15]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class FarmEnv(gym.Env):
    """
    Custom Reinforcement Learning Environment for Farmer Crop Planning
    """

    def __init__(self, df):
        super(FarmEnv, self).__init__()

        # Store dataset
        self.df = df.reset_index(drop=True)

        # Starting row
        self.current_step = 0

        # Initial farmer savings
        self.initial_savings = 100000

        # List of available crops
        self.crops = self.df["Crop"].unique()

        # Action Space
        self.action_space = spaces.Discrete(len(self.crops))

        # Observation Space
        # [Rainfall, Market Price, Fertilizer, Pesticide, Yield, Savings]
        self.observation_space = spaces.Box(
            low=0,
            high=np.inf,
            shape=(6,),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.savings = self.initial_savings

        row = self.df.iloc[self.current_step]

        state = np.array([
            row["Annual_Rainfall"],
            row["Modal_Price"],
            row["Fertilizer"],
            row["Pesticide"],
            row["Yield"],
            self.savings
        ], dtype=np.float32)

        return state, {}

    def step(self, action):

        row = self.df.iloc[self.current_step]

        # Selected crop
        selected_crop = self.crops[action]

        # Reward = Profit
        reward = row["Profit"]

        # Update savings
        self.savings += reward

        # Move to next record
        self.current_step += 1

        # Check if episode finished
        done = self.current_step >= len(self.df) - 1

        if not done:

            next_row = self.df.iloc[self.current_step]

            next_state = np.array([
                next_row["Annual_Rainfall"],
                next_row["Modal_Price"],
                next_row["Fertilizer"],
                next_row["Pesticide"],
                next_row["Yield"],
                self.savings
            ], dtype=np.float32)

        else:

            next_state = np.zeros(6, dtype=np.float32)

        info = {
            "Selected Crop": selected_crop,
            "Profit": reward,
            "Savings": self.savings
        }

        return next_state, reward, done, False, info

    def render(self):

        print(f"Step : {self.current_step}")
        print(f"Savings : {self.savings}")


In [16]:
env = FarmEnv(df)

print("Environment Created Successfully!")

Environment Created Successfully!


In [17]:
state, info = env.reset()

print("Initial State")

print(state)

Initial State
[2.0513999e+03 7.0189521e+03 1.6550062e+05 5.3909003e+02 4.2090908e-01
 1.0000000e+05]


In [18]:
next_state, reward, done, truncated, info = env.step(0)

print("Reward :", reward)

print("Done :", done)

print("Info :", info)

print("Next State :")

print(next_state)

Reward : 5407008.211506849
Done : False
Info : {'Selected Crop': 'Cotton', 'Profit': np.float64(5407008.211506849), 'Savings': np.float64(5507008.211506849)}
Next State :
[2.0513999e+03 2.6000000e+03 1.8287868e+06 5.9569600e+03 6.1565220e-01
 5.5070080e+06]


In [19]:
state, info = env.reset()

for i in range(5):

    action = env.action_space.sample()

    next_state, reward, done, truncated, info = env.step(action)

    print("="*40)

    print("Step :", i+1)

    print("Action :", action)

    print("Crop :", info["Selected Crop"])

    print("Reward :", reward)

    print("Savings :", info["Savings"])

    if done:
        print("Episode Finished")
        break

Step : 1
Action : 1
Crop : Maize
Reward : 5407008.211506849
Savings : 5507008.211506849
Step : 2
Action : 3
Crop : Potato
Reward : 36439856.32
Savings : 41946864.53150685
Step : 3
Action : 3
Crop : Potato
Reward : 25531660.540310346
Savings : 67478525.07181719
Step : 4
Action : 2
Crop : Onion
Reward : 618962558.8710167
Savings : 686441083.9428339
Step : 5
Action : 5
Crop : Banana
Reward : 262912932.60220185
Savings : 949354016.5450357
